# RS-VLM Phase 2 — Kaggle Version

Before running:
1. Go to the **Settings** panel on the right.
2. Turn **Internet** ON.
3. Set **Accelerator** to GPU T4 x2 (or P100).
4. Click **Add Input** (top right) -> **Your Datasets** -> Add `rsicd-dataset-custom` and `rs-vlm-checkpoints`.

In [ ]:
!nvidia-smi

## 1. Setup & Install

In [ ]:
import os

if not os.path.exists('/kaggle/working/rs_vlm'):
    !git clone https://github.com/sharksurfauto-byte/rs_vlm.git /kaggle/working/rs_vlm
else:
    !git -C /kaggle/working/rs_vlm pull
    !find /kaggle/working/rs_vlm -name '*.pyc' -delete

%cd /kaggle/working/rs_vlm
!pip install -q -r requirements.txt
print('Ready.')

## 2. Configure Paths for Kaggle

In [ ]:
import yaml, sys, os
sys.path.insert(0, '/kaggle/working/rs_vlm')

# IMPORTANT: Verify these match your Kaggle Input folder names!
RSICD_DIR = '/kaggle/input/rsicd-dataset-custom'
P1_CKPT = '/kaggle/input/rs-vlm-checkpoints/encoder_final.pt'
P2_CKPT_DIR = '/kaggle/working/checkpoints/phase2'

os.makedirs(P2_CKPT_DIR, exist_ok=True)

with open('/kaggle/working/rs_vlm/configs/colab_config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data']['rsicd_root']      = RSICD_DIR
cfg['data']['num_workers']     = 2  
cfg['phase2']['batch_size']    = 8
cfg['phase2']['epochs']        = 10
cfg['phase2']['checkpoint_dir']= P2_CKPT_DIR

with open('/kaggle/working/rs_vlm/configs/colab_config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print("Config Patched for Kaggle.")

## 3. Run Phase 2 Training

In [ ]:
from training.phase2_projector import train_phase2

model = train_phase2(
    config_path='configs/colab_config.yaml',
    phase1_checkpoint=P1_CKPT,
    resume_from=None,  # Update this if resuming from a crashed run
)